# 7.14 少量錯誤答案會被學走嗎？


你故意把1+1答案寫成3，交叉熵會知道人類標錯了嗎？不會。它只衡量模型是否相信我們交給它的目標，沒有內建算術裁判。若把錯答案給高機率，錯標資料的loss就會下降。因此訓練分數很好，可能表示正確示範學好了，也可能把錯誤一起記住。

[1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html) 的loss只取提供label那一格的機率。本節縮成四個候選「0、1、2、3」，ID也恰好0到3，方便對比真實答案2與錯標籤3。這是獨立的小分類表，不是ByteTokenizer的數字字符ID，避免和 [7.11](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html) 的byte答案映射混淆。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch.nn import functional as F

logits = torch.tensor([[0.0, 0.0, 3.0, 1.0]])
for label in [2, 3]:
    loss = F.cross_entropy(logits, torch.tensor([label]))
    print("提供的答案ID", label, "代價", loss.item())
wrong_favored = torch.tensor([[0.0, 0.0, 3.0, 5.0]])
print("提高錯字分數後，錯標代價", F.cross_entropy(wrong_favored, torch.tensor([3])).item())

輸入四分數，真實候選2得3、錯候選3得1。softmax後2約0.810，3約0.110，所以提供label2時代價約0.211、label3時約2.211。最後把錯候選分數提高到5，錯誤標籤的代價降到約0.139，儘管真實答案2的相對支持變少。交叉熵完整遵從輸入label，沒有自動檢查1+1的數學事實。

真實訓練中，錯誤答案也會產生清楚梯度，推動這類前文更常接錯字。如果錯例反覆出現或答案很長，其有效token影響可能超出筆數比例，見 [5.16](https://birdhackor.github.io/tiny-perceptron-vlm/5.16.html)。少量錯誤有沒有可觀影響還要實測，不能只靠這一格代價給出所有模型都會如何的量化結論。

比較錯標影響時，應保持樣本數、問題、更新步數與種子相同，只換某部分答案。種子是隨機數產生器的起始設定，用來控制開始權重與抽樣等隨機選擇；兩次保持相同，有助於比較標籤改動，而不是不同隨機起點。這能區分資料變多與標籤改變。

評估要使用答案已獨立核對的留出題，也就是在訓練前保留、沒有參與更新模型的新題，分組原則見[1.2](https://birdhackor.github.io/tiny-perceptron-vlm/1.2.html)。例如訓練用過的1+1另按正解2檢查，是在檢查舊題；預先保留且沒拿來訓練的2+1，再按正解3檢查，才是在看新題。兩種結果分開保存，才能知道影響只限於原問題，還是延伸到新問題。不要把同一錯標訓練集的低loss當成正確率證據。

資料品質檢查因此不是可有可無的裝飾：明確可驗證任務先核對答案，開放式示例則需檢查是否支持事實、是否足夠資訊與是否一致。模型也可能有已有知識抵抗少量錯誤，但這不能代替資料負責人確認label。

練習只把最後錯候選分數5改成8，先預測錯標loss繼續下降、但真答案的loss變大，再執行並另算label2核對。這次只改模型偏好，沒有更改算術事實，能直接看到「優化提供目標」與「外部任務正確」的差別。
